# 02 — Regression Models for Absorbance PredictionThis notebook trains and evaluates regression models to predictmean peptide absorbance from sequence-derived features.Models explored: Ridge, Random Forest, Gradient Boosting, Stacking Ensemble.Feature strategies: sequence-only, sequence + dipeptides, PCA-compressed hybrid.

In [ ]:
import sys, ossys.path.insert(0, os.path.abspath('..'))import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.model_selection import train_test_split, GridSearchCVfrom sklearn.preprocessing import StandardScalerfrom sklearn.decomposition import PCAfrom sklearn.linear_model import Ridgefrom sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressorfrom sklearn.feature_selection import SelectKBest, f_regressionfrom src.feature_extraction import (    extract_features_dataframe,    extract_dipeptide_features,    validate_sequence,)from src.models import (    evaluate_regression,    compare_models,    tune_random_forest,    create_stacking_regressor,)from src.visualization import (    plot_actual_vs_predicted,    plot_feature_importance,)%matplotlib inline

## 1. Load Processed Data

In [ ]:
DATA_DIR = os.path.join('..', 'data')df_model = pd.read_csv(os.path.join(DATA_DIR, 'processed_features.csv'))# Identify feature columns (exclude metadata and target)skip_cols = {'Sequence', 'mean_abs', 'std_abs', 'cv', 'max_abs', 'min_abs'}seq_features = [c for c in df_model.columns if c not in skip_cols]print(f'Dataset: {df_model.shape[0]} sequences, {len(seq_features)} features')df_model.head()

## 2. Baseline Model Comparison (Sequence Features Only)

In [ ]:
# Top features identified from correlation analysistop_features = [    'negative_charge_ratio', 'aa_percent_E', 'flex_mean', 'aa_percent_D',    'charged_ratio', 'instability_index', 'n_term_polar', 'turn_frac', 'mol_weight',]X = df_model[top_features]y = df_model['mean_abs']X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)scaler = StandardScaler()X_train_scaled = scaler.fit_transform(X_train)X_test_scaled = scaler.transform(X_test)

In [ ]:
models = {    'Ridge': Ridge(),    'RandomForest': RandomForestRegressor(random_state=42),    'GradientBoosting': GradientBoostingRegressor(random_state=42),}results = compare_models(models, X_train_scaled, y_train, X_test_scaled, y_test)results

## 3. Dipeptide Features + PCA HybridExtract 400 dipeptide frequency features, compress via PCA,and combine with top sequence features.

In [ ]:
# Extract dipeptide featuresdf_dipeptides = pd.DataFrame([    extract_dipeptide_features(validate_sequence(seq))    for seq in df_model['Sequence']])print(f'Dipeptide features: {df_dipeptides.shape}')# PCA compressionpca = PCA(n_components=50, random_state=42)X_dipep_pca = pca.fit_transform(df_dipeptides)df_pca = pd.DataFrame(X_dipep_pca, columns=[f'PCA_{i+1}' for i in range(50)])print(f'Explained variance (50 components): {pca.explained_variance_ratio_.sum():.3f}')

In [ ]:
# Hybrid feature setdf_hybrid = pd.concat([    df_model[top_features].reset_index(drop=True),    df_pca.reset_index(drop=True),], axis=1)X_hybrid = df_hybrid.valuesy = df_model['mean_abs'].valuesscaler_h = StandardScaler()X_hybrid_scaled = scaler_h.fit_transform(X_hybrid)X_train_h, X_test_h, y_train, y_test = train_test_split(    X_hybrid_scaled, y, test_size=0.2, random_state=42)

In [ ]:
# Compare models with hybrid featuresmodels_h = {    'Ridge': Ridge(),    'RandomForest': RandomForestRegressor(random_state=42),    'GradientBoosting': GradientBoostingRegressor(random_state=42),}results_hybrid = compare_models(models_h, X_train_h, y_train, X_test_h, y_test)results_hybrid

## 4. Hyperparameter Tuning (Random Forest)

In [ ]:
grid = tune_random_forest(X_hybrid_scaled, y, cv=5, scoring='neg_mean_squared_error')print(f'Best params: {grid.best_params_}')best_rf = grid.best_estimator_y_pred_best = best_rf.predict(X_test_h)metrics = evaluate_regression(y_test, y_pred_best)print(f"Tuned RF — R²: {metrics['r2']:.4f}, RMSE: {metrics['rmse']:.4f}, MAE: {metrics['mae']:.4f}")

## 5. Stacking Ensemble

In [ ]:
stack = create_stacking_regressor(best_rf=best_rf)stack.fit(X_train_h, y_train)y_pred_stack = stack.predict(X_test_h)metrics_stack = evaluate_regression(y_test, y_pred_stack)print(f"Stacking — R²: {metrics_stack['r2']:.4f}, RMSE: {metrics_stack['rmse']:.4f}, MAE: {metrics_stack['mae']:.4f}")

## 6. Visualize Best Model

In [ ]:
fig = plot_actual_vs_predicted(y_test, y_pred_best, title='Tuned RF: Actual vs Predicted')plt.show()

In [ ]:
if hasattr(best_rf, 'feature_importances_'):    fig = plot_feature_importance(        best_rf.feature_importances_,        list(df_hybrid.columns),        n_top=20,    )    plt.show()

## 7. Predict New Sequences

In [ ]:
def predict_new_peptides(peptides, scaler, pca_model, model, top_features_list, pca_cols):    """Predict absorbance for a list of new peptide sequences."""    from src.feature_extraction import extract_all_features, extract_dipeptide_features, validate_sequence    rows = []    for seq in peptides:        seq_clean = validate_sequence(seq)        feats = extract_all_features(seq)        dipep = extract_dipeptide_features(seq_clean)        dipep_pca = pca_model.transform([list(dipep.values())])        pca_dict = {f'PCA_{i+1}': dipep_pca[0][i] for i in range(dipep_pca.shape[1])}        row = {f: feats.get(f, 0) for f in top_features_list}        row.update(pca_dict)        rows.append(row)    X_new = pd.DataFrame(rows)    X_new_scaled = scaler.transform(X_new)    preds = model.predict(X_new_scaled)    return pd.DataFrame({'Sequence': peptides, 'Predicted_Absorbance': preds})test_peptides = [    'SKENSHSQAINVDRD',    'KYENSHSQAINVDDT',    'SYDDSHSQAIKKDRT',    'SYEKSHKQDINVDRD',]predict_new_peptides(test_peptides, scaler_h, pca, best_rf, top_features, df_pca.columns)